In [2]:
from Data_query.trino_config import *
import json
import numpy as np
import matplotlib.pyplot as plt
from visualisation import *
import pytz

In [ ]:
sleep(60)
import subprocess as sp
sp.run("shutdown -h now", shell=True)

In [ ]:
stop_trino()


Trino service stopping triggered.


In [18]:
big_workers = 1
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count = workers, big_worker_desired_count=big_workers)
sleep(30)


Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [3]:
df = iceberg_sql('select * from conformance_sust_op')

In [7]:
df.query('year ==2024 and month==1 and total_count > 5 and nonconformance_sust_op_count == 0')

,year,month,day,day_night,site_id,nonconformance_sust_op_sum,nonconformance_sust_op_count,total_count,v_threshold
552832,2024,1,1,day,63908987,0.0,0,29,258.0
552962,2024,1,31,day,1629237411,0.0,0,27,258.0
558314,2024,1,14,day,266954605,0.0,0,22,258.0
558315,2024,1,15,day,266954605,0.0,0,15,258.0
558316,2024,1,16,day,266954605,0.0,0,19,258.0
...,...,...,...,...,...,...,...,...,...
587394,2024,1,11,day,299607190,0.0,0,8,253.0
587399,2024,1,17,day,299607190,0.0,0,6,253.0
587547,2024,1,18,day,961389838,0.0,0,7,253.0
587578,2024,1,13,day,1188137514,0.0,0,10,253.0


In [20]:
iceberg_exec("DROP TABLE IF EXISTS conformance_sust_op")
iceberg_exec("""CREATE TABLE conformance_sust_op (
                year INT,
                month INT,
                day INT,
                day_night VARCHAR,
                site_id BIGINT,
                nonconformance_sust_op_sum DOUBLE,
                nonconformance_sust_op_count BIGINT,
                total_count BIGINT,
                v_threshold DOUBLE
            )
    """)

Executed
Executed


In [ ]:
def run_func(args):
    year, month, v_threshold, split_cons = args
    iceberg_exec(f"""insert into conformance_sust_op
                        with data as (
                            select site_id, t_stamp, sum(cast(power*circuit_polarity as decimal(18, 6)))/1000 as P_kW, 
                                    max(cast(voltage as decimal(18, 6))) as V, ac_capacity_kw
                        from 
                        (select circuit_id, t_stamp, power, energy_reactive, voltage 
                        from ts where year={year} and month={month} and is_pv = true and voltage > 0 and voltage < 350 
                        and {split_cons} 
                        ) as ts1
                        inner join (select site_id, circuit_id, circuit_polarity,  ac_capacity_kw from meta_up23c 
                        ) as m on ts1.circuit_id = m.circuit_id
                        group by site_id, t_stamp,  ac_capacity_kw),
                        data1 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw,
                        lag(t_stamp) over (partition by site_id order by t_stamp) as prev_t_stamp,
                        lag(t_stamp, 2) over (partition by site_id order by t_stamp) as prev_t_stamp_2,
                        lag(V) over (partition by site_id order by t_stamp) as prev_V,
                        lag(V, 2) over (partition by site_id order by t_stamp) as prev_V_2,
                        lag(P_kW) over (partition by site_id order by t_stamp) as prev_P_kW,
                        lag(P_kW, 2) over (partition by site_id order by t_stamp) as prev_P_kW_2
                        from data
                        ),
                        data2 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw, day(t_stamp) as day,
                        case when (hour(t_stamp) >= 20 or hour(t_stamp) <= 7) then 'day' else 'night' end as day_night,
                        case when P_kW >= 0.04 * ac_capacity_kw then P_kW - 0.04 * ac_capacity_kw else 0 end as nonconformance_sum
                        from data1
                        where V >= {v_threshold} and prev_V >= {v_threshold} 
                        and (prev_V_2 >= {v_threshold} or prev_P_kW_2 > 0.04 * ac_capacity_kw or P_kW > 0.04 * ac_capacity_kw))

                        select {year} as year, {month} as month, day, day_night, site_id,  
                        sum(nonconformance_sum) as nonconformance_sust_op_sum, 
                        sum(case when nonconformance_sum > 0 then 1 else 0 end) as nonconformance_sust_op_count,
                        count(nonconformance_sum) as total_count,
                        {v_threshold} as v_threshold
                        from data2
                        where day_night = 'day'
                        group by site_id, day, day_night
                        """)
                        # 
    
    print(f"Completed year={year}, month={month}, v_threshold={v_threshold}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}")
    return None

tasks = [(year, month, v_threshold, split_cons) for year in (2024, 2025) for month in range(1, 13) for v_threshold in range(253, 259) 
         for split_cons in ['system.bucket(postcode, 16) <= 3', '(system.bucket(postcode, 16) > 3 and system.bucket(postcode, 16) <= 7)', 
                            '(system.bucket(postcode, 16) > 7 and system.bucket(postcode, 16) <= 11)', 'system.bucket(postcode, 16) > 11'] ]
        #  for split_cons in ['system.bucket(postcode, 16) <= 1'] ]
         
df2 = trino_parallel(run_func, tasks, num_workers=num_workers)

Executed
Completed year=2024, month=1, v_threshold=253, bucket <= 3
Executed
Completed year=2024, month=1, v_threshold=253, (bucket > 3 and bucket <= 7)
Executed
Completed year=2024, month=1, v_threshold=253, (bucket > 7 and bucket <= 11)
Executed
Completed year=2024, month=1, v_threshold=253, bucket > 11
Executed
Completed year=2024, month=1, v_threshold=254, bucket <= 3
Executed
Completed year=2024, month=1, v_threshold=254, (bucket > 3 and bucket <= 7)
Executed
Completed year=2024, month=1, v_threshold=254, (bucket > 7 and bucket <= 11)
Executed
Completed year=2024, month=1, v_threshold=254, bucket > 11
Executed
Completed year=2024, month=1, v_threshold=255, bucket <= 3
Executed
Completed year=2024, month=1, v_threshold=255, (bucket > 3 and bucket <= 7)
Executed
Completed year=2024, month=1, v_threshold=255, (bucket > 7 and bucket <= 11)
Executed
Completed year=2024, month=1, v_threshold=255, bucket > 11
Executed
Completed year=2024, month=1, v_threshold=256, bucket <= 3
Executed
Co

In [ ]:
v_threshold = 258
def run_func(args):
    year, month, v_threshold, split_cons = args
    df = iceberg_sql(f"""
                        with data as (
                            select site_id, t_stamp, sum(cast(power*circuit_polarity as decimal(18, 6)))/1000 as P_kW, 
                                    max(cast(voltage as decimal(18, 6))) as V, ac_capacity_kw
                        from 
                        (select circuit_id, t_stamp, power, energy_reactive, voltage 
                        from ts where year={year} and month={month} and is_pv = true and voltage > 0 and voltage < 350 
                        and {split_cons} 
                        ) as ts1
                        inner join (select site_id, circuit_id, circuit_polarity,  ac_capacity_kw from meta_up23c 
                        ) as m on ts1.circuit_id = m.circuit_id
                        group by site_id, t_stamp,  ac_capacity_kw),
                        data1 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw,
                        lag(t_stamp) over (partition by site_id order by t_stamp) as prev_t_stamp,
                        lag(t_stamp, 2) over (partition by site_id order by t_stamp) as prev_t_stamp_2,
                        lag(V) over (partition by site_id order by t_stamp) as prev_V,
                        lag(V, 2) over (partition by site_id order by t_stamp) as prev_V_2,
                        lag(P_kW) over (partition by site_id order by t_stamp) as prev_P_kW,
                        lag(P_kW, 2) over (partition by site_id order by t_stamp) as prev_P_kW_2
                        from data
                        ),
                        data2 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw, day(t_stamp) as day,
                        case when (hour(t_stamp) >= 20 or hour(t_stamp) <= 7) then 'day' else 'night' end as day_night,
                        case when P_kW >= 0.04 * ac_capacity_kw then P_kW - 0.04 * ac_capacity_kw else 0 end as nonconformance_sum
                        from data1
                        where V >= {v_threshold} and prev_V >= {v_threshold} 
                        and (prev_V_2 >= {v_threshold} or prev_P_kW_2 > 0.04 * ac_capacity_kw or P_kW > 0.04 * ac_capacity_kw))
                        select {year} as year, {month} as month, site_id,  day, day_night,

                        sum(nonconformance_sum) as nonconformance_sum, 
                        sum(case when nonconformance_sum > 0 then 1 else 0 end) as nonconformance_count,
                        count(nonconformance_sum) as total_count,
                        {v_threshold} as v_threshold
                        from data2
                        where day_night = 'day'
                        group by site_id, day, day_night
                        """)
                        # 
    
    print(f"Completed year={year}, month={month}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}")
    return df

tasks = [(year, month, split_cons) for year in (2024, ) for month in range(1, 2) 
         for split_cons in ['system.bucket(postcode, 16) <= 3', '(system.bucket(postcode, 16) > 3 and system.bucket(postcode, 16) <= 7)', 
                            '(system.bucket(postcode, 16) > 7 and system.bucket(postcode, 16) <= 11)', 'system.bucket(postcode, 16) > 11'] ]
        #  for split_cons in ['system.bucket(postcode, 16) <= 1'] ]
         
df2 = trino_parallel(run_func, tasks, num_workers=num_workers)

Completed year=2024, month=1, bucket <= 3
Completed year=2024, month=1, (bucket > 3 and bucket <= 7)
Completed year=2024, month=1, (bucket > 7 and bucket <= 11)
Completed year=2024, month=1, bucket > 11


In [11]:
df2

,year,month,site_id,day,day_night,nonconformance_sum,nonconformance_count,total_count,v_threshold
0,2024,1,1992537319,5,night,0.000000,0,13,258
1,2024,1,609703575,4,day,94.425900,10,10,258
2,2024,1,609703575,5,day,151.259014,16,16,258
3,2024,1,609703575,8,day,18.973936,2,2,258
4,2024,1,609703575,9,day,9.488813,1,1,258
...,...,...,...,...,...,...,...,...,...
224,2024,1,221985833,28,day,6.213926,2,2,258
225,2024,1,221985833,29,day,21.235136,5,5,258
226,2024,1,221985833,30,day,4.001363,1,1,258
227,2024,1,1108512667,10,day,28.734913,3,3,258


In [16]:
df.query("P_kW >= 0.04 * ac_capacity_kw")

,year,month,site_id,t_stamp,t_stamp_next,V,ac_capacity_kw,P_kW,P_kW_next
19,2024,2,1559806445,2024-02-01 01:05:00,2024-02-01 01:10:00,265.20,10.0,8.548817,8.609913
20,2024,2,1559806445,2024-02-01 01:10:00,2024-02-01 01:15:00,266.65,10.0,8.609913,8.620447
21,2024,2,1559806445,2024-02-01 01:15:00,2024-02-01 01:20:00,266.60,10.0,8.620447,8.449297
22,2024,2,1559806445,2024-02-01 01:40:00,2024-02-01 01:45:00,265.90,10.0,8.278790,8.055660
23,2024,2,1559806445,2024-02-01 02:00:00,2024-02-01 02:05:00,266.15,10.0,8.590050,8.460400
...,...,...,...,...,...,...,...,...,...
1,2025,5,615902948,2025-05-03 02:25:00,2025-05-03 02:30:00,265.73,10.0,9.942690,9.871970
2,2025,5,615902948,2025-05-03 02:30:00,2025-05-03 02:35:00,265.05,10.0,9.871970,9.970260
3,2025,5,615902948,2025-05-03 02:35:00,2025-05-03 02:40:00,265.16,10.0,9.970260,9.899350
4,2025,5,615902948,2025-05-03 02:40:00,2025-05-03 02:45:00,265.35,10.0,9.899350,9.967260


In [14]:
df['site_id'].nunique()

234